# 03 – Metrics layer
One view per metric, built from clean tables and following metric_definitions.md. The dashboard and the Genie space read only these views.
Reporting month: August 2026 (latest complete month). Trends from January 2023.

## Step 1 – Month calendar and MAU

In [0]:
-- Month calendar 
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_months 
COMMENT 'One row per calendar month, January 2021 to August 2026' AS 
SELECT month_start
, LAST_DAY(month_start) AS month_end
FROM (
    SELECT EXPLODE(SEQUENCE(DATE'2021-01-01', DATE'2026-08-01', INTERVAL 1 MONTH)) AS month_start
);

In [0]:
-- MAU View
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_mau (
  month_start COMMENT 'First day of the calendar month',
  business_id COMMENT 'BIZ_1, BIZ_2, BIZ_3, or All businesses. Business MAUs do not add up to the total: search events and events on test content have no business, and a user active in two businesses counts in both',
  mau COMMENT 'Distinct regular users with at least one qualifying event in the month'
)
COMMENT 'MAU (Monthly Active Users): distinct regular users with at least one qualifying event in the calendar month. Qualifying events: content_open, content_complete, like, share, comment_post, download, search_query. Excludes page_view, subscription events, test accounts and quarantined events. Includes regular users events on test content (flag on_test_content). Latest complete month: August 2026.'
AS
WITH qualifying_events AS (
  SELECT user_id,
         content_id,
         CAST(DATE_TRUNC('MONTH', event_timestamp) AS DATE) AS month_start
  FROM workspace.scribd_exercise.clean_engagement_events
  WHERE event_type IN ('content_open', 'content_complete', 'like', 'share',
                       'comment_post', 'download', 'search_query')
),
total AS (
  SELECT month_start,
         'All businesses' AS business_id,
         COUNT(DISTINCT user_id) AS mau
  FROM qualifying_events
  GROUP BY month_start
),
by_business AS (
  SELECT q.month_start,
         c.business_id,
         COUNT(DISTINCT q.user_id) AS mau
  FROM qualifying_events q
  JOIN workspace.scribd_exercise.clean_content_items c
    ON q.content_id = c.content_id
  GROUP BY q.month_start, c.business_id
),
all_rows AS (
  SELECT * FROM total
  UNION ALL
  SELECT * FROM by_business
),
businesses AS (
  SELECT 'All businesses' AS business_id
  UNION ALL
  SELECT business_id FROM workspace.scribd_exercise.clean_businesses
)
SELECT m.month_start,
       b.business_id,
       COALESCE(a.mau, 0) AS mau --Use the MAU or 0 if there is none
FROM workspace.scribd_exercise.metrics_months m
CROSS JOIN businesses b
LEFT JOIN all_rows a
  ON a.month_start = m.month_start
 AND a.business_id = b.business_id;

In [0]:
--Check MAU View
SELECT month_start, business_id, mau
FROM workspace.scribd_exercise.metrics_mau
WHERE month_start >= DATE'2026-05-01'
ORDER BY month_start, business_id;

month_start,business_id,mau
2026-05-01,All businesses,255
2026-05-01,BIZ_1,84
2026-05-01,BIZ_2,77
2026-05-01,BIZ_3,127
2026-06-01,All businesses,296
2026-06-01,BIZ_1,109
2026-06-01,BIZ_2,102
2026-06-01,BIZ_3,127
2026-07-01,All businesses,354
2026-07-01,BIZ_1,138


## Step 2 – Active subscribers

In [0]:
-- Helper: current subscription per user, business and month end 
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_current_subscriptions (
  month_end COMMENT 'Last day of the month the subscription is evaluated on',
  user_id COMMENT 'Regular user',
  business_id COMMENT 'Business subscribed to',
  subscription_id COMMENT 'The current subscription for this user and business on month_end',
  plan_tier COMMENT 'Plan of the current subscription',
  billing_cycle COMMENT 'monthly, annual, lifetime_deal_fake (a monthly price locked for life), or null for Free',
  monthly_price_usd COMMENT 'Monthly price; placeholder prices already replaced with the grid price',
  is_paid COMMENT '1 if the plan is not Free',
  dq_flags COMMENT 'Data-quality flags of the subscription'
)
COMMENT 'Helper: for each month end, each user current subscription per business. Active on a date = started on or before it, not ended by it, and current status not paused, trialing or unknown. If several are active, paid is preferred over Free, then the most recent. Status is a current snapshot, so earlier months are approximate.'
AS
WITH active_on_date AS (
  SELECT m.month_end, s.*
  FROM workspace.scribd_exercise.metrics_months m
  JOIN workspace.scribd_exercise.clean_subscriptions s
    ON s.start_date <= m.month_end
   AND (s.end_date IS NULL OR s.end_date > m.month_end)
  WHERE s.status NOT IN ('paused', 'trialing', 'unknown')
),
ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY month_end, user_id, business_id
           ORDER BY CASE WHEN plan_tier <> 'Free' THEN 1 ELSE 0 END DESC,
           start_date DESC,
           subscription_id DESC
         ) AS rn
  FROM active_on_date
)
SELECT month_end
, user_id
, business_id
, subscription_id
, plan_tier
, billing_cycle
, monthly_price_usd
, CASE WHEN plan_tier <> 'Free' THEN 1 ELSE 0 END AS is_paid
, dq_flags
FROM ranked
WHERE rn = 1;

In [0]:
-- Active Subscribers View 
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_active_subscribers (
  month_end COMMENT 'Date the count is taken on (last day of the month)',
  business_id COMMENT 'BIZ_1, BIZ_2, BIZ_3, or All businesses (distinct people: someone paying for two businesses counts once). Business counts do not add up to the total',
  active_paid_subscribers COMMENT 'Distinct users with an active paid subscription on month_end (to that business, or to any business on the All businesses row)',
  active_free_users COMMENT 'Distinct users with a current Free-plan subscription (to that business, or to any business on the All businesses row)',
  upper_bound_paid_subscribers COMMENT 'Upper bound of the range: also counts users with a paid subscription whose status cannot be trusted (unknown status, or a trial older than 30 days), if not already counted. Recent trials are excluded as genuine. Meaningful for the latest month only'
)
COMMENT 'Active subscribers: distinct regular users with at least one active paid subscription on the date. Active = started on or before the date, not ended by it, and current status not paused, trialing or unknown. All trials are excluded because trial users are not paying yet. Excludes test accounts. Upper bound: adds paid subscriptions whose status cannot be trusted, i.e. unknown status (issue #16) and stale trials older than 30 days that may have converted without the status being updated (issue #19). Recent trials (30 days or less) are not in the upper bound: they are genuine trials in progress. Headline: as of 2026-08-31. Status is a current snapshot, so earlier months are approximate.'
AS
WITH current_subs AS (
  SELECT * FROM workspace.scribd_exercise.metrics_current_subscriptions
),
-- Paid subscriptions whose status can't be trusted: started by the date, not ended
uncertain AS (
  SELECT m.month_end, s.user_id, s.business_id
  FROM workspace.scribd_exercise.metrics_months m
  JOIN workspace.scribd_exercise.clean_subscriptions s
    ON s.start_date <= m.month_end
   AND s.end_date IS NULL
  WHERE s.plan_tier <> 'Free'
    AND (s.status = 'unknown'
         OR (s.status = 'trialing' AND s.dq_flags LIKE '%stale_trial%'))
),
paying AS (
  SELECT month_end, user_id, business_id
  FROM current_subs
  WHERE is_paid = 1
),

-- One row per business and month
by_business AS (
  SELECT m.month_end,
         b.business_id,
         (SELECT COUNT(DISTINCT p.user_id) FROM paying p
           WHERE p.month_end = m.month_end AND p.business_id = b.business_id) AS active_paid_subscribers,
         (SELECT COUNT(DISTINCT c.user_id) FROM current_subs c
           WHERE c.month_end = m.month_end AND c.business_id = b.business_id AND c.is_paid = 0) AS active_free_users,
         (SELECT COUNT(DISTINCT x.user_id) FROM (
             SELECT user_id FROM paying
              WHERE month_end = m.month_end AND business_id = b.business_id
             UNION
             SELECT user_id FROM uncertain
              WHERE month_end = m.month_end AND business_id = b.business_id
          ) x) AS upper_bound_paid_subscribers
  FROM workspace.scribd_exercise.metrics_months m
  CROSS JOIN workspace.scribd_exercise.clean_businesses b
),

-- One row per month for all businesses: distinct people
all_businesses AS (
  SELECT m.month_end,
         'All businesses' AS business_id,
         (SELECT COUNT(DISTINCT p.user_id) FROM paying p
           WHERE p.month_end = m.month_end) AS active_paid_subscribers,
         (SELECT COUNT(DISTINCT c.user_id) FROM current_subs c
           WHERE c.month_end = m.month_end AND c.is_paid = 0) AS active_free_users,
         (SELECT COUNT(DISTINCT x.user_id) FROM (
             SELECT user_id FROM paying WHERE month_end = m.month_end
             UNION
             SELECT user_id FROM uncertain WHERE month_end = m.month_end
          ) x) AS upper_bound_paid_subscribers
  FROM workspace.scribd_exercise.metrics_months m
)

SELECT * FROM by_business
UNION ALL
SELECT * FROM all_businesses;

In [0]:
-- Check
SELECT *
FROM workspace.scribd_exercise.metrics_active_subscribers
WHERE month_end >= DATE'2026-06-30'
ORDER BY month_end, business_id;

month_end,business_id,active_paid_subscribers,active_free_users,upper_bound_paid_subscribers
2026-06-30,All businesses,275,99,300
2026-06-30,BIZ_1,87,33,96
2026-06-30,BIZ_2,106,35,117
2026-06-30,BIZ_3,96,35,107
2026-07-31,All businesses,279,105,306
2026-07-31,BIZ_1,87,37,96
2026-07-31,BIZ_2,104,34,117
2026-07-31,BIZ_3,101,38,112
2026-08-31,All businesses,285,109,314
2026-08-31,BIZ_1,92,36,103


**Result:** 
- As of 2026-08-31: 285 paying people (range 285–314, the upper bound adding unknown-status subscriptions and stale trials)
- 109 users on Free plans. 
- By business: BIZ_1 92, BIZ_2 108, BIZ_3 99; they sum to 299 because 14 people pay for two businesses. 
- Paying people grew from 275 (June) to 285 (August).

## Step 3 – MRR
Sum of the monthly price of each user's current paid subscription per business, at every month end. All businesses = sum of the businesses. Split by billing cycle, with an upper bound for subscriptions whose status cannot be trusted.

In [0]:
-- MRR View
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_mrr (
  month_end COMMENT 'Date MRR is measured on (last day of the month)',
  business_id COMMENT 'BIZ_1, BIZ_2, BIZ_3, or All businesses (sum of the businesses: revenue adds up, so business rows add up to the total)',
  mrr_usd COMMENT 'Monthly recurring revenue: sum of the monthly price of each user current paid subscription',
  mrr_monthly_usd COMMENT 'Part of MRR from monthly-billed subscriptions',
  mrr_annual_usd COMMENT 'Part of MRR from annual subscriptions (prices are already monthly amounts)',
  mrr_lifetime_usd COMMENT 'Part of MRR from lifetime deals (billing_cycle lifetime_deal_fake), treated as a monthly price locked for life (issue #17)',
  mrr_upper_bound_usd COMMENT 'Upper bound of the range: adds paid subscriptions with unknown status or trials older than 30 days, for users without a current paid subscription to that business. Meaningful for the latest month only'
)
COMMENT 'MRR (Monthly Recurring Revenue): sum of the monthly price of each regular user current paid subscription per business, as of the month end. Current = active on the date (started, not ended, status not paused, trialing or unknown), one per user and business (paid preferred, then most recent; issue #20). Placeholder prices replaced with the grid price (#18). Lifetime deals included and shown separately (#17). Listed prices, not collected revenue (no payment data). Headline: as of 2026-08-31. Status is a current snapshot, so earlier months are approximate.'
AS
WITH current_paid AS (
  SELECT *
  FROM workspace.scribd_exercise.metrics_current_subscriptions
  WHERE is_paid = 1
),
-- Paid subscriptions whose status can't be trusted, one per user and business
uncertain AS (
  SELECT m.month_end, s.user_id, s.business_id, s.monthly_price_usd,
         ROW_NUMBER() OVER (
           PARTITION BY m.month_end, s.user_id, s.business_id
           ORDER BY s.start_date DESC, s.subscription_id DESC
         ) AS rn
  FROM workspace.scribd_exercise.metrics_months m
  JOIN workspace.scribd_exercise.clean_subscriptions s
    ON s.start_date <= m.month_end
   AND s.end_date IS NULL
  WHERE s.plan_tier <> 'Free'
    AND (s.status = 'unknown'
         OR (s.status = 'trialing' AND s.dq_flags LIKE '%stale_trial%'))
),
-- Only for users not already paying for that business on that date
uncertain_extra AS (
  SELECT u.month_end, u.business_id, SUM(u.monthly_price_usd) AS extra_usd
  FROM uncertain u
  LEFT ANTI JOIN current_paid c
    ON c.month_end = u.month_end
   AND c.business_id = u.business_id
   AND c.user_id = u.user_id
  WHERE u.rn = 1
  GROUP BY u.month_end, u.business_id
),
by_business AS (
  SELECT month_end,
         business_id,
         SUM(monthly_price_usd) AS mrr_usd,
         SUM(CASE WHEN billing_cycle = 'monthly' THEN monthly_price_usd ELSE 0 END) AS mrr_monthly_usd,
         SUM(CASE WHEN billing_cycle = 'annual' THEN monthly_price_usd ELSE 0 END) AS mrr_annual_usd,
         SUM(CASE WHEN billing_cycle = 'lifetime_deal_fake' THEN monthly_price_usd ELSE 0 END) AS mrr_lifetime_usd
  FROM current_paid
  GROUP BY month_end, business_id
),
business_rows AS (
  SELECT m.month_end,
         b.business_id,
         ROUND(COALESCE(bb.mrr_usd, 0), 2) AS mrr_usd,
         ROUND(COALESCE(bb.mrr_monthly_usd, 0), 2) AS mrr_monthly_usd,
         ROUND(COALESCE(bb.mrr_annual_usd, 0), 2) AS mrr_annual_usd,
         ROUND(COALESCE(bb.mrr_lifetime_usd, 0), 2) AS mrr_lifetime_usd,
         ROUND(COALESCE(bb.mrr_usd, 0) + COALESCE(ue.extra_usd, 0), 2) AS mrr_upper_bound_usd
  FROM workspace.scribd_exercise.metrics_months m
  CROSS JOIN workspace.scribd_exercise.clean_businesses b
  LEFT JOIN by_business bb
    ON bb.month_end = m.month_end AND bb.business_id = b.business_id
  LEFT JOIN uncertain_extra ue
    ON ue.month_end = m.month_end AND ue.business_id = b.business_id
)
SELECT * FROM business_rows
UNION ALL
SELECT month_end,
       'All businesses',
       ROUND(SUM(mrr_usd), 2),
       ROUND(SUM(mrr_monthly_usd), 2),
       ROUND(SUM(mrr_annual_usd), 2),
       ROUND(SUM(mrr_lifetime_usd), 2),
       ROUND(SUM(mrr_upper_bound_usd), 2)
FROM business_rows
GROUP BY month_end;

In [0]:
SELECT *
FROM workspace.scribd_exercise.metrics_mrr
WHERE month_end >= DATE'2026-06-30'
ORDER BY month_end, business_id;

month_end,business_id,mrr_usd,mrr_monthly_usd,mrr_annual_usd,mrr_lifetime_usd,mrr_upper_bound_usd
2026-06-30,All businesses,3759.11,1269.24,1550.76,939.11,4179.8
2026-06-30,BIZ_1,1077.13,359.77,427.65,289.71,1190.04
2026-06-30,BIZ_2,1430.94,484.69,611.52,334.73,1594.83
2026-06-30,BIZ_3,1251.04,424.78,511.59,314.67,1394.93
2026-07-31,All businesses,3805.08,1324.22,1546.76,934.1,4255.75
2026-07-31,BIZ_1,1091.13,409.75,391.67,289.71,1204.04
2026-07-31,BIZ_2,1403.96,449.71,639.51,314.74,1597.83
2026-07-31,BIZ_3,1309.99,464.76,515.58,329.65,1453.88
2026-08-31,All businesses,3956.01,1414.2,1582.73,959.08,4441.65
2026-08-31,BIZ_1,1191.08,439.75,431.64,319.69,1330.97


**Result:** 
- MRR as of 2026-08-31 = $3,956.01 (range up to $4,441.65 with unknown-status subscriptions and stale trials)
- Up 4% from July ($3,805.08). 
- By business: BIZ_1 $1,191.08, BIZ_2 $1,484.92, BIZ_3 $1,280.01. 
- Lifetime deals are 24% of MRR ($959.08); excluding them, MRR would be $2,996.93. 
- Average revenue per paying person: about $13.88/month.

## Step 4 – Churn (trailing 3 months)
Share of paying customers per business at the start of a 3-month window who no longer pay at the end because their paid subscription ended during it. Plan or billing switches are not churn. Split by business and billing cycle.

In [0]:
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_churn (
  window_start COMMENT 'First day of the 3-month window',
  window_end COMMENT 'Last day of the 3-month window (headline: 2026-08-31, i.e. June to August 2026)',
  business_id COMMENT 'BIZ_1, BIZ_2, BIZ_3, or All businesses (pooled per business: a customer leaving one business but keeping another counts as churn for the one they left)',
  billing_cycle COMMENT 'Billing cycle of the customer subscription at the start of the window, or All billing cycles',
  starting_customers COMMENT 'Customers paying for the business at the end of the month before the window (e.g. May 31 for June to August)',
  churned_customers COMMENT 'Of those, customers not paying at the window end whose paid subscription ended during the window',
  churn_rate COMMENT 'churned_customers / starting_customers. Rate over the 3-month window, not a monthly rate',
  churn_rate_ci_low COMMENT 'Lower end of the 95% confidence interval for churn_rate (Wilson interval). Wide intervals mean small counts',
  churn_rate_ci_high COMMENT 'Upper end of the 95% confidence interval for churn_rate (Wilson interval). If two groups intervals overlap a lot, their difference may be chance'
)
COMMENT 'Subscription churn rate, trailing 3 months: customers paying for a business at the end of the month before the window who are no longer paying at its end because their paid subscription ended during it. Switching plan or billing cycle is not churn; new customers who joined during the window are not counted. Includes a 95% Wilson confidence interval, since counts per segment are small. Source: subscriptions table only (cancellation events are unreliable, issue #24). May be understated if some statuses were never updated (issue #19). Status is a current snapshot, so earlier windows are approximate.'
AS
WITH windows AS (
  SELECT month_end AS window_end,
         ADD_MONTHS(month_start, -2) AS window_start,
         LAST_DAY(ADD_MONTHS(month_start, -3)) AS start_snapshot
  FROM workspace.scribd_exercise.metrics_months
  WHERE month_start >= DATE'2021-04-01'
),
paid AS (
  SELECT month_end, user_id, business_id, billing_cycle
  FROM workspace.scribd_exercise.metrics_current_subscriptions
  WHERE is_paid = 1
),
starting AS (
  SELECT w.window_start, w.window_end, p.user_id, p.business_id, p.billing_cycle
  FROM windows w
  JOIN paid p ON p.month_end = w.start_snapshot
),
still_paying AS (
  SELECT w.window_end, p.user_id, p.business_id
  FROM windows w
  JOIN paid p ON p.month_end = w.window_end
),
ended_in_window AS (
  SELECT DISTINCT w.window_end, s.user_id, s.business_id
  FROM windows w
  JOIN workspace.scribd_exercise.clean_subscriptions s
    ON s.end_date BETWEEN w.window_start AND w.window_end
  WHERE s.plan_tier <> 'Free'
),
marked AS (
  SELECT st.*,
         CASE WHEN sp.user_id IS NULL AND e.user_id IS NOT NULL THEN 1 ELSE 0 END AS churned
  FROM starting st
  LEFT JOIN still_paying sp
    ON sp.window_end = st.window_end AND sp.user_id = st.user_id AND sp.business_id = st.business_id
  LEFT JOIN ended_in_window e
    ON e.window_end = st.window_end AND e.user_id = st.user_id AND e.business_id = st.business_id
),
agg AS (
  SELECT window_start,
         window_end,
         COALESCE(business_id, 'All businesses') AS business_id,
         COALESCE(billing_cycle, 'All billing cycles') AS billing_cycle,
         COUNT(*) AS n,
         SUM(churned) AS k
  FROM marked
  GROUP BY GROUPING SETS (
    (window_start, window_end, business_id, billing_cycle),
    (window_start, window_end, business_id),
    (window_start, window_end, billing_cycle),
    (window_start, window_end)
  )
)
SELECT window_start,
       window_end,
       business_id,
       billing_cycle,
       n AS starting_customers,
       k AS churned_customers,
       ROUND(k / n, 4) AS churn_rate,
       -- 95% Wilson interval (z = 1.96)
       ROUND(((k / n) + 1.96 * 1.96 / (2 * n)
              - 1.96 * SQRT((k / n) * (1 - k / n) / n + 1.96 * 1.96 / (4 * n * n)))
             / (1 + 1.96 * 1.96 / n), 4) AS churn_rate_ci_low,
       ROUND(((k / n) + 1.96 * 1.96 / (2 * n)
              + 1.96 * SQRT((k / n) * (1 - k / n) / n + 1.96 * 1.96 / (4 * n * n)))
             / (1 + 1.96 * 1.96 / n), 4) AS churn_rate_ci_high
FROM agg;

In [0]:
SELECT business_id, billing_cycle, starting_customers, churned_customers, churn_rate, churn_rate_ci_low, churn_rate_ci_high
FROM workspace.scribd_exercise.metrics_churn
WHERE window_end = DATE'2026-08-31'
ORDER BY business_id, billing_cycle;

business_id,billing_cycle,starting_customers,churned_customers,churn_rate,churn_rate_ci_low,churn_rate_ci_high
All businesses,All billing cycles,287,38,0.1324,0.098,0.1765
All businesses,annual,123,12,0.0976,0.0567,0.1628
All businesses,lifetime_deal_fake,83,4,0.0482,0.0189,0.1175
All businesses,monthly,81,22,0.2716,0.1867,0.3772
BIZ_1,All billing cycles,89,14,0.1573,0.0961,0.2469
BIZ_1,annual,36,5,0.1389,0.0608,0.2866
BIZ_1,lifetime_deal_fake,26,0,0.0,0.0,0.1287
BIZ_1,monthly,27,9,0.3333,0.1864,0.5218
BIZ_2,All billing cycles,108,14,0.1296,0.0788,0.2059
BIZ_2,annual,49,4,0.0816,0.0322,0.1919


**Result:** 
- Churn for June–August 2026 = 13.2% over 3 months (38 of 287 customers; about 4.6% a month). 
- By billing cycle: monthly 27.2%, annual 9.8%, lifetime 4.8%. 
- By business: BIZ_1 15.7%, BIZ_2 13.0%, BIZ_3 11.1%. 
- Some segments are small (e.g., BIZ_1 lifetime 0 of 26), so the billing-cycle comparison is most reliable across all businesses.

## Step 5 – Content engagement share by content age
For each month: share of content engagement going to new (0–90 days), recent (91–365) and older (366+) content, next to each band's share of the catalog and engagement per item.

In [0]:
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_content_age (
  month_start COMMENT 'First day of the calendar month',
  business_id COMMENT 'BIZ_1, BIZ_2, BIZ_3, or All businesses',
  age_band COMMENT 'Content age: New (0-90 days), Recent (91-365 days), Older (366+ days). For engagement: age at the event; for the catalog: age at month end',
  band_order COMMENT 'Sort order of the age bands (1 = New)',
  content_events COMMENT 'Content events in the month on content in this age band',
  engagement_share COMMENT 'content_events / all content events in the month (same business)',
  live_items COMMENT 'Live catalog items in this age band at month end',
  catalog_share COMMENT 'live_items / all live catalog items at month end (same business)',
  engagement_per_item COMMENT 'content_events / live_items. Compare across bands: lower per item for new content is a lead to investigate (quality or discovery), not a conclusion'
)
COMMENT 'Content engagement share by content age, monthly. Content events = content_open, content_complete, like, share, comment_post, download on clean catalog items (test content excluded). Events dated before their content was published are excluded (issue #6). Live catalog = items published on or before month end (assumes items stay available). The New band holds few items per month, so it is volatile.'
AS
WITH bands AS (
  SELECT * FROM VALUES
    ('New (0-90 days)', 1),
    ('Recent (91-365 days)', 2),
    ('Older (366+ days)', 3)
  AS t(age_band, band_order)
),
content_events AS (
  SELECT CAST(DATE_TRUNC('MONTH', e.event_timestamp) AS DATE) AS month_start,
         c.business_id,
         CASE WHEN DATEDIFF(CAST(e.event_timestamp AS DATE), c.publish_date) <= 90 THEN 'New (0-90 days)'
              WHEN DATEDIFF(CAST(e.event_timestamp AS DATE), c.publish_date) <= 365 THEN 'Recent (91-365 days)'
              ELSE 'Older (366+ days)' END AS age_band
  FROM workspace.scribd_exercise.clean_engagement_events e
  JOIN workspace.scribd_exercise.clean_content_items c
    ON e.content_id = c.content_id
  WHERE e.event_type IN ('content_open', 'content_complete', 'like', 'share', 'comment_post', 'download')
    AND CAST(e.event_timestamp AS DATE) >= c.publish_date
),
events_agg AS (
  SELECT month_start,
         COALESCE(business_id, 'All businesses') AS business_id,
         age_band,
         COUNT(*) AS content_events
  FROM content_events
  GROUP BY GROUPING SETS ((month_start, business_id, age_band), (month_start, age_band))
),
catalog AS (
  SELECT m.month_start,
         c.business_id,
         CASE WHEN DATEDIFF(m.month_end, c.publish_date) <= 90 THEN 'New (0-90 days)'
              WHEN DATEDIFF(m.month_end, c.publish_date) <= 365 THEN 'Recent (91-365 days)'
              ELSE 'Older (366+ days)' END AS age_band
  FROM workspace.scribd_exercise.metrics_months m
  JOIN workspace.scribd_exercise.clean_content_items c
    ON c.publish_date <= m.month_end
),
catalog_agg AS (
  SELECT month_start,
         COALESCE(business_id, 'All businesses') AS business_id,
         age_band,
         COUNT(*) AS live_items
  FROM catalog
  GROUP BY GROUPING SETS ((month_start, business_id, age_band), (month_start, age_band))
),
grid AS (
  SELECT m.month_start, b.business_id, bd.age_band, bd.band_order
  FROM workspace.scribd_exercise.metrics_months m
  CROSS JOIN (SELECT 'All businesses' AS business_id
              UNION ALL SELECT business_id FROM workspace.scribd_exercise.clean_businesses) b
  CROSS JOIN bands bd
),
combined AS (
  SELECT g.month_start, g.business_id, g.age_band, g.band_order,
         COALESCE(ea.content_events, 0) AS content_events,
         COALESCE(ca.live_items, 0) AS live_items
  FROM grid g
  LEFT JOIN events_agg ea
    ON ea.month_start = g.month_start AND ea.business_id = g.business_id AND ea.age_band = g.age_band
  LEFT JOIN catalog_agg ca
    ON ca.month_start = g.month_start AND ca.business_id = g.business_id AND ca.age_band = g.age_band
)
SELECT month_start,
       business_id,
       age_band,
       band_order,
       content_events,
       ROUND(content_events / NULLIF(SUM(content_events) OVER (PARTITION BY month_start, business_id), 0), 4) AS engagement_share,
       live_items,
       ROUND(live_items / NULLIF(SUM(live_items) OVER (PARTITION BY month_start, business_id), 0), 4) AS catalog_share,
       ROUND(content_events / NULLIF(live_items, 0), 2) AS engagement_per_item
FROM combined;

In [0]:
SELECT business_id, age_band, content_events, engagement_share, live_items, catalog_share, engagement_per_item
FROM workspace.scribd_exercise.metrics_content_age
WHERE month_start = DATE'2026-08-01'
ORDER BY business_id, band_order;

business_id,age_band,content_events,engagement_share,live_items,catalog_share,engagement_per_item
All businesses,New (0-90 days),176,0.2924,22,0.0769,8.0
All businesses,Recent (91-365 days),197,0.3272,65,0.2273,3.03
All businesses,Older (366+ days),229,0.3804,199,0.6958,1.15
BIZ_1,New (0-90 days),34,0.1848,4,0.0381,8.5
BIZ_1,Recent (91-365 days),50,0.2717,16,0.1524,3.13
BIZ_1,Older (366+ days),100,0.5435,85,0.8095,1.18
BIZ_2,New (0-90 days),13,0.0909,2,0.025,6.5
BIZ_2,Recent (91-365 days),53,0.3706,15,0.1875,3.53
BIZ_2,Older (366+ days),77,0.5385,63,0.7875,1.22
BIZ_3,New (0-90 days),129,0.4691,16,0.1584,8.06


**Result:** 
- In August 2026, new content (0–90 days) got more engagement per item than older content in every business (6.5–8.5 events per item vs. 1.0–1.2). 
- But only BIZ_3 publishes enough new content for it to drive most engagement (47%)
- BIZ_1 and BIZ_2 have very few new items (4 and 2), so most of their engagement comes from older content (54% each). 
- Leads: whether BIZ_1 and BIZ_2 under-supply new content, and how discoverable the back catalog is. Association, not cause; new-item counts are small.

## Step 6 – Catalog reach
Share of the live catalog at month end with at least one content event in the month. Total, by business, category, content type, and runtime within content type, with a 95% confidence interval.

In [0]:
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_catalog_reach (
  month_start COMMENT 'First day of the calendar month',
  business_id COMMENT 'BIZ_1, BIZ_2, BIZ_3, or All businesses',
  dimension COMMENT 'What the row is broken down by: total, category, content type, or content age',
  dimension_value COMMENT 'The value of the dimension: All, a category name, a content type, or a content age band (New 0-90 days, Recent 91-365 days, Older 366+ days, measured at month end)',
  live_items COMMENT 'Live catalog items at month end (published on or before it)',
  reached_items COMMENT 'Of those, items with at least one content event from a regular user in the month',
  catalog_reach COMMENT 'reached_items / live_items',
  catalog_reach_ci_low COMMENT 'Lower end of the 95% confidence interval for catalog_reach (Wilson interval). Wide intervals mean few items',
  catalog_reach_ci_high COMMENT 'Upper end of the 95% confidence interval for catalog_reach (Wilson interval). If two groups intervals overlap a lot, their difference may be chance'
)
COMMENT 'Catalog reach, monthly: share of live catalog items with at least one content event (open, complete, like, share, comment, download) in the month. Live catalog = clean items published on or before month end (test content excluded; assumes items stay available). Events dated before publishing are excluded. Broken down by business, category, content type and content age at month end. A visibility measure: how much of the catalog gets seen. Includes a 95% Wilson confidence interval.'
AS
WITH engaged AS (
  -- Items with at least one content event in each month
  SELECT DISTINCT
         CAST(DATE_TRUNC('MONTH', e.event_timestamp) AS DATE) AS month_start,
         e.content_id
  FROM workspace.scribd_exercise.clean_engagement_events e
  JOIN workspace.scribd_exercise.clean_content_items c
    ON e.content_id = c.content_id
  WHERE e.event_type IN ('content_open', 'content_complete', 'like', 'share', 'comment_post', 'download')
    AND CAST(e.event_timestamp AS DATE) >= c.publish_date
),
items AS (
  -- One row per live item per month, marked reached (1) or not (0)
  SELECT m.month_start,
         c.content_id,
         c.business_id,
         c.category,
         c.content_type,
         CASE WHEN DATEDIFF(m.month_end, c.publish_date) <= 90 THEN 'New (0-90 days)'
              WHEN DATEDIFF(m.month_end, c.publish_date) <= 365 THEN 'Recent (91-365 days)'
              ELSE 'Older (366+ days)' END AS age_band,
         CASE WHEN en.content_id IS NOT NULL THEN 1 ELSE 0 END AS reached
  FROM workspace.scribd_exercise.metrics_months m
  JOIN workspace.scribd_exercise.clean_content_items c
    ON c.publish_date <= m.month_end
  LEFT JOIN engaged en
    ON en.month_start = m.month_start AND en.content_id = c.content_id
),
agg AS (
  SELECT month_start,
         COALESCE(business_id, 'All businesses') AS business_id,
         CASE WHEN GROUPING(age_band) = 0 THEN 'content age'
              WHEN GROUPING(category) = 0 THEN 'category'
              WHEN GROUPING(content_type) = 0 THEN 'content type'
              ELSE 'total' END AS dimension,
         CASE WHEN GROUPING(age_band) = 0 THEN age_band
              WHEN GROUPING(category) = 0 THEN category
              WHEN GROUPING(content_type) = 0 THEN content_type
              ELSE 'All' END AS dimension_value,
         COUNT(*) AS n,
         SUM(reached) AS k
  FROM items
  GROUP BY GROUPING SETS (
    (month_start),
    (month_start, business_id),
    (month_start, business_id, category),
    (month_start, content_type),
    (month_start, business_id, content_type),
    (month_start, age_band),
    (month_start, business_id, age_band)
  )
)
SELECT month_start,
       business_id,
       dimension,
       dimension_value,
       n AS live_items,
       k AS reached_items,
       ROUND(k / n, 4) AS catalog_reach,
       -- 95% Wilson interval (z = 1.96)
       ROUND(((k / n) + 1.96 * 1.96 / (2 * n)
              - 1.96 * SQRT((k / n) * (1 - k / n) / n + 1.96 * 1.96 / (4 * n * n)))
             / (1 + 1.96 * 1.96 / n), 4) AS catalog_reach_ci_low,
       ROUND(((k / n) + 1.96 * 1.96 / (2 * n)
              + 1.96 * SQRT((k / n) * (1 - k / n) / n + 1.96 * 1.96 / (4 * n * n)))
             / (1 + 1.96 * 1.96 / n), 4) AS catalog_reach_ci_high
FROM agg;

In [0]:
SELECT business_id, dimension, dimension_value, live_items, reached_items,
       catalog_reach, catalog_reach_ci_low, catalog_reach_ci_high
FROM workspace.scribd_exercise.metrics_catalog_reach
WHERE month_start = DATE'2026-08-01'
  AND dimension IN ('total', 'content type','content age')
ORDER BY business_id, dimension DESC, catalog_reach DESC;

business_id,dimension,dimension_value,live_items,reached_items,catalog_reach,catalog_reach_ci_low,catalog_reach_ci_high
All businesses,total,All,286,216,0.7552,0.7022,0.8015
All businesses,content type,poll,46,39,0.8478,0.7178,0.9243
All businesses,content type,serialized_chapter,51,40,0.7843,0.6537,0.8751
All businesses,content type,audiobook,49,38,0.7755,0.6412,0.8698
All businesses,content type,forum_post,53,38,0.717,0.5843,0.8203
All businesses,content type,video,41,29,0.7073,0.5552,0.8239
All businesses,content type,ebook,46,32,0.6957,0.5519,0.8092
All businesses,content age,New (0-90 days),22,22,1.0,0.8513,1.0
All businesses,content age,Recent (91-365 days),65,62,0.9538,0.8729,0.9842
All businesses,content age,Older (366+ days),199,132,0.6633,0.5951,0.7253


**Results:** 
- In August 2026, 75.5% of the live catalog (216 of 286 items) was seen at least once (95% CI 70–80%). 
-By content age: new 100% (22 of 22), recent 95% (62 of 65), older 66% (132 of 199, CI 60–73%). 67 of the 70 unseen items are older content, so low visibility is concentrated in the back catalog.
- By business: BIZ_3 85%, BIZ_2 72.5%, BIZ_1 69%. BIZ_1's lower reach reflects its catalog mix (81% older content), not clearly lower visibility of its older items (62% vs. 67% and 73%, intervals overlap).
- Differences by content type are within chance.

In [0]:
SELECT business_id, dimension_value, live_items, reached_items,
       catalog_reach, catalog_reach_ci_low, catalog_reach_ci_high
FROM workspace.scribd_exercise.metrics_catalog_reach
WHERE month_start = DATE'2026-08-01'
  AND dimension = 'content age'
ORDER BY business_id, dimension_value;

## Step 7 – Metadata completeness
Share of the live catalog with all required metadata valid, plus each requirement's pass rate. A data-health metric for content understanding.

In [0]:
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_metadata_completeness (
  month_start COMMENT 'First day of the month; the catalog is evaluated at month end',
  business_id COMMENT 'BIZ_1, BIZ_2, BIZ_3, or All businesses',
  requirement COMMENT 'All requirements, or one requirement: Valid content type, Category present, Valid runtime, Creator linked (creator_content only)',
  items_checked COMMENT 'Live catalog items the requirement applies to (Creator linked applies only to the creator_content business line)',
  items_passing COMMENT 'Of those, items that meet the requirement',
  pass_rate COMMENT 'items_passing / items_checked. For All requirements: share of items with complete, valid metadata'
)
COMMENT 'Metadata completeness: share of the live catalog at month end whose required metadata is present and valid: valid content type, category present, valid runtime (not a placeholder, issue #7), and creator linked for creator_content items (issue #1). Measures presence and validity, not correctness (a wrong category would still pass).'
AS
WITH items AS (
  SELECT m.month_start,
         c.content_id,
         c.business_id,
         CASE WHEN c.content_type IS NOT NULL AND c.content_type <> 'TEST_TYPE' THEN 1 ELSE 0 END AS type_ok,
         CASE WHEN c.category IS NOT NULL AND TRIM(c.category) <> '' THEN 1 ELSE 0 END AS category_ok,
         CASE WHEN c.runtime_minutes IS NOT NULL THEN 1 ELSE 0 END AS runtime_ok,
         -- Creator only applies to the creator_content business line; NULL = not applicable
         CASE WHEN b.business_line <> 'creator_content' THEN NULL
              WHEN c.creator_id IS NOT NULL THEN 1 ELSE 0 END AS creator_ok
  FROM workspace.scribd_exercise.metrics_months m
  JOIN workspace.scribd_exercise.clean_content_items c
    ON c.publish_date <= m.month_end
  JOIN workspace.scribd_exercise.clean_businesses b
    ON c.business_id = b.business_id
),
checks AS (
  SELECT month_start, business_id, 'All requirements' AS requirement,
         CASE WHEN type_ok = 1 AND category_ok = 1 AND runtime_ok = 1
                   AND (creator_ok IS NULL OR creator_ok = 1) THEN 1 ELSE 0 END AS passed
  FROM items
  UNION ALL
  SELECT month_start, business_id, 'Valid content type', type_ok FROM items
  UNION ALL
  SELECT month_start, business_id, 'Category present', category_ok FROM items
  UNION ALL
  SELECT month_start, business_id, 'Valid runtime', runtime_ok FROM items
  UNION ALL
  SELECT month_start, business_id, 'Creator linked (creator_content only)', creator_ok
  FROM items
  WHERE creator_ok IS NOT NULL
)
SELECT month_start,
       COALESCE(business_id, 'All businesses') AS business_id,
       requirement,
       COUNT(*) AS items_checked,
       SUM(passed) AS items_passing,
       ROUND(SUM(passed) / COUNT(*), 4) AS pass_rate
FROM checks
GROUP BY GROUPING SETS ((month_start, business_id, requirement), (month_start, requirement));

In [0]:
SELECT business_id, requirement, items_checked, items_passing, pass_rate
FROM workspace.scribd_exercise.metrics_metadata_completeness
WHERE month_start = DATE'2026-08-01'
ORDER BY business_id, requirement;

business_id,requirement,items_checked,items_passing,pass_rate
All businesses,All requirements,286,276,0.965
All businesses,Category present,286,286,1.0
All businesses,Creator linked (creator_content only),101,101,1.0
All businesses,Valid content type,286,286,1.0
All businesses,Valid runtime,286,276,0.965
BIZ_1,All requirements,105,98,0.9333
BIZ_1,Category present,105,105,1.0
BIZ_1,Valid content type,105,105,1.0
BIZ_1,Valid runtime,105,98,0.9333
BIZ_2,All requirements,80,77,0.9625


**Result:** 
- 96.5% of the live catalog (276 of 286 items) has complete, valid metadata as of August 2026.
- All failures are invalid runtimes (10 items: 7 in BIZ_1, 3 in BIZ_2)
- Content type, category and creator links pass 100%
- BIZ_3 is fully complete.

## Step 8 – Headline view
Every headline metric for the latest complete month in one view, with ranges or confidence intervals where they apply.

In [0]:
CREATE OR REPLACE VIEW workspace.scribd_exercise.metrics_headline (
  metric COMMENT 'Metric name',
  business_id COMMENT 'BIZ_1, BIZ_2, BIZ_3, or All businesses',
  period COMMENT 'Period the number refers to (latest complete month; churn uses the 3 months ending then)',
  value COMMENT 'The metric value (rates as decimals, e.g. 0.75 = 75%)',
  range_low COMMENT 'Lower end: 95% confidence interval for rates, or the headline value for counts with an upper bound',
  range_high COMMENT 'Upper end: 95% confidence interval for rates, or the upper bound for active subscribers and MRR',
  unit COMMENT 'Unit of the value'
)
COMMENT 'Headline metrics for the latest complete month (August 2026 at the time of building), one row per metric and business. Reads from the metric views, so definitions are the same as in each view. Ranges: active subscribers and MRR include an upper bound for subscriptions whose status cannot be trusted; churn and catalog reach include 95% confidence intervals.'
AS
WITH latest AS (
  SELECT MAX(month_start) AS month_start, MAX(month_end) AS month_end
  FROM workspace.scribd_exercise.metrics_months
)
SELECT 'MAU' AS metric, m.business_id,
       DATE_FORMAT(l.month_start, 'MMMM yyyy') AS period,
       CAST(m.mau AS DOUBLE) AS value, CAST(NULL AS DOUBLE) AS range_low, CAST(NULL AS DOUBLE) AS range_high,
       'users' AS unit
FROM workspace.scribd_exercise.metrics_mau m
JOIN latest l ON m.month_start = l.month_start

UNION ALL
SELECT 'Active paid subscribers', a.business_id,
       CONCAT('as of ', CAST(l.month_end AS STRING)),
       a.active_paid_subscribers, a.active_paid_subscribers, a.upper_bound_paid_subscribers,
       'people'
FROM workspace.scribd_exercise.metrics_active_subscribers a
JOIN latest l ON a.month_end = l.month_end

UNION ALL
SELECT 'MRR', r.business_id,
       CONCAT('as of ', CAST(l.month_end AS STRING)),
       r.mrr_usd, r.mrr_usd, r.mrr_upper_bound_usd,
       'USD per month'
FROM workspace.scribd_exercise.metrics_mrr r
JOIN latest l ON r.month_end = l.month_end

UNION ALL
SELECT 'MRR from lifetime deals', r.business_id,
       CONCAT('as of ', CAST(l.month_end AS STRING)),
       r.mrr_lifetime_usd, NULL, NULL,
       'USD per month'
FROM workspace.scribd_exercise.metrics_mrr r
JOIN latest l ON r.month_end = l.month_end

UNION ALL
SELECT 'Churn rate (3 months)', c.business_id,
       CONCAT(DATE_FORMAT(c.window_start, 'MMM'), '–', DATE_FORMAT(c.window_end, 'MMM yyyy')),
       c.churn_rate, c.churn_rate_ci_low, c.churn_rate_ci_high,
       'share of starting customers'
FROM workspace.scribd_exercise.metrics_churn c
JOIN latest l ON c.window_end = l.month_end
WHERE c.billing_cycle = 'All billing cycles'

UNION ALL
SELECT 'Engagement share of new content (0-90 days)', a.business_id,
       DATE_FORMAT(l.month_start, 'MMMM yyyy'),
       a.engagement_share, NULL, NULL,
       'share of content events'
FROM workspace.scribd_exercise.metrics_content_age a
JOIN latest l ON a.month_start = l.month_start
WHERE a.age_band = 'New (0-90 days)'

UNION ALL
SELECT 'Catalog share of new content (0-90 days)', a.business_id,
       DATE_FORMAT(l.month_start, 'MMMM yyyy'),
       a.catalog_share, NULL, NULL,
       'share of live catalog'
FROM workspace.scribd_exercise.metrics_content_age a
JOIN latest l ON a.month_start = l.month_start
WHERE a.age_band = 'New (0-90 days)'

UNION ALL
SELECT 'Catalog reach', cr.business_id,
       DATE_FORMAT(l.month_start, 'MMMM yyyy'),
       cr.catalog_reach, cr.catalog_reach_ci_low, cr.catalog_reach_ci_high,
       'share of live catalog'
FROM workspace.scribd_exercise.metrics_catalog_reach cr
JOIN latest l ON cr.month_start = l.month_start
WHERE cr.dimension = 'total'

UNION ALL
SELECT 'Metadata completeness', mc.business_id,
       DATE_FORMAT(l.month_start, 'MMMM yyyy'),
       mc.pass_rate, NULL, NULL,
       'share of live catalog'
FROM workspace.scribd_exercise.metrics_metadata_completeness mc
JOIN latest l ON mc.month_start = l.month_start
WHERE mc.requirement = 'All requirements';

In [0]:
SELECT metric, period, value, range_low, range_high, unit
FROM workspace.scribd_exercise.metrics_headline
WHERE business_id = 'All businesses'
ORDER BY metric;

metric,period,value,range_low,range_high,unit
Active paid subscribers,as of 2026-08-31,285.0,285.0,314.0,people
Catalog reach,August 2026,0.7552,0.7022,0.8015,share of live catalog
Catalog share of new content (0-90 days),August 2026,0.0769,null,null,share of live catalog
Churn rate (3 months),Jun–Aug 2026,0.1324,0.098,0.1765,share of starting customers
Engagement share of new content (0-90 days),August 2026,0.2924,null,null,share of content events
MAU,August 2026,401.0,null,null,users
MRR,as of 2026-08-31,3956.01,3956.01,4441.65,USD per month
MRR from lifetime deals,as of 2026-08-31,959.08,null,null,USD per month
Metadata completeness,August 2026,0.965,null,null,share of live catalog
